# Memetic algorithms — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Combine population search with local improvement so candidates inherit better habits
A memetic algorithm adds local search to an evolutionary loop. The population explores broadly, but each promising candidate is refined by a problem-specific improvement step before it competes.

In [ ]:
# 1) Local search can sharply improve one evolutionary candidate
x=2.0; eta=0.2; grad=2*(x-3); x2=x-eta*grad; before=(x-3)**2; after=(x2-3)**2
xs=np.linspace(0,5,100); plt.figure(figsize=(6,3)); plt.plot(xs,(xs-3)**2); plt.scatter([x,x2],[before,after],c=['tab:red','tab:green']); plt.arrow(x,before,x2-x,after-before,head_width=.08,length_includes_head=True); plt.title('one local step refines a candidate')
assert abs(x2-2.4)<1e-12 and abs(after-0.36)<1e-12

In [ ]:
# 2) Evolution supplies candidates; local search polishes the selected one
candidates=np.array([0.,2.,5.]); vals=(candidates-3)**2; best=candidates[np.argmin(vals)]; polished=best-0.2*2*(best-3)
plt.figure(figsize=(6,3)); plt.bar(['0','2','5','polished 2'], list(vals)+[(polished-3)**2]); plt.ylabel('objective'); plt.title('selection finds a basin, local search descends it')
assert best==2.0 and (polished-3)**2 < vals.min()

In [ ]:
# 3) Memetic search beats GA-only on a small 1D objective
rng=np.random.default_rng(13)
def run(memetic):
    pop=rng.uniform(-5,5,25); hist=[]
    for _ in range(25):
        val=(pop-3)**2+0.15*np.sin(5*pop); hist.append(val.min()); probs=softmax_min(val, temp=1.0); p=rng.choice(pop,(25,2),p=probs); child=p.mean(axis=1)+rng.normal(0,0.4,25)
        if memetic:
            grad=2*(child-3)+0.75*np.cos(5*child); child=child-0.08*grad
        pop=child
    return hist
rng=np.random.default_rng(13); ga=run(False); rng=np.random.default_rng(13); mem=run(True)
plt.figure(figsize=(6,3)); plt.semilogy(ga,label='GA only'); plt.semilogy(mem,label='memetic'); plt.legend(); plt.ylabel('best objective'); plt.xlabel('generation'); plt.title('local improvement accelerates convergence')
assert mem[-1] < ga[-1]

In [ ]:
# 4) Too much local search can collapse diversity
rng=np.random.default_rng(14); rates=[0.0,0.5,1.0]; div=[]
for rate in rates:
    pop=rng.uniform(-5,5,40)
    for _ in range(12):
        val=(pop-3)**2; p=rng.choice(pop,(40,2),p=softmax_min(val,1.0)); pop=p.mean(axis=1)+rng.normal(0,0.5,40); mask=rng.random(40)<rate; pop[mask]=pop[mask]-0.25*2*(pop[mask]-3)
    div.append(pop.std())
plt.figure(figsize=(6,3)); plt.plot(rates,div,'-o'); plt.xlabel('local-search rate'); plt.ylabel('population std'); plt.title('memes improve candidates but reduce variety')
assert div[-1] < div[0]

In [ ]:
# 5) In 2D, local polishing tightens the final population around the optimum
rng=np.random.default_rng(15); target=np.array([1.,-1.]); pop=rng.uniform(-4,4,(40,2)); start=pop.copy()
for _ in range(18):
    val=np.sum((pop-target)**2,axis=1); p=rng.choice(len(pop),(40,2),p=softmax_min(val,2.0)); child=(pop[p[:,0]]+pop[p[:,1]])/2+rng.normal(0,0.35,(40,2)); child=child-0.3*2*(child-target); pop=child
plt.figure(figsize=(4,4)); plt.scatter(start[:,0],start[:,1],alpha=.25,label='start'); plt.scatter(pop[:,0],pop[:,1],alpha=.8,label='final'); plt.scatter([1],[-1],c='r',marker='*',s=120); plt.legend(); plt.axis('equal'); plt.title('population search plus local refinement')
assert np.linalg.norm(pop.mean(0)-target) < np.linalg.norm(start.mean(0)-target)